# HLA Inference from TCRβ Repertoires — THNet on Russell 2022 Cohort

**Method**: THNet v1.0.3 (Pan et al. 2025) — HLA typing from TCRβ bulk sequencing using sequence-embedding-based classifiers trained on 4,144 HLA-genotyped individuals.

**Dataset**: Russell et al. 2022 (PRJNA762269 / Zenodo DOI:10.5281/zenodo.5719516) — 149 donors from the Nicaraguan Influenza Cohort Study with paired TCRα+TCRβ immunosequencing and HLA ground truth (SNP-based genotyping).

**Key distinction from Rosati**: This is the first cohort in this project with **HLA ground truth**, enabling quantitative validation of inference accuracy (sensitivity, specificity, PPV, AUC-ROC per allele).

**Population context**: Russell donors are Nicaraguan (Latin American ancestry). THNet was trained predominantly on European/American cohorts. This population mismatch is a central finding of this analysis.

**Reference**: Pan et al. 2025, *Quantitative and large-scale investigation of human TCR-HLA cross-reactivity*, GitHub: github.com/Mia-yao/THNet  
**Data**: Russell et al. 2022, *Combining genotypes and T cell receptor distributions to infer genetic loci determining V(D)J recombination probabilities*, eLife.

## 0. Setup & paths

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import time
import warnings
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from THNet.HLA_inference.model_prediction import Model_prediction
from THNet.HLA_inference import HLA_inference

warnings.filterwarnings('ignore')

BASE             = '/home/immunologylab/bioinformatics/'
RUSSELL_DIR      = BASE + 'raw_data/russell_hla/nicaragua_data/'
HLA_FILE         = BASE + 'soft/miniconda3/envs/bioinf/lib/python3.12/site-packages/HLAGuessr/Training_data/HLA_grouped_patients.tsv'
RESULTS          = BASE + 'hla-tcr-project/results/'
CROSSVAL_CSV     = RESULTS + 'hlaguessr_46B_crossval.csv'
RUSSELL_THNET_OUT = RESULTS + 'thnet_russell_output/'

os.makedirs(RUSSELL_THNET_OUT, exist_ok=True)

def norm_vgene_thnet(v):
    """TRBV5-1*01 → TRBV5-1 (full gene without allele, THNet format)"""
    if pd.isna(v): return None
    return str(v).split('*')[0].strip()

## 1. Dataset overview & HLA ground truth

The Russell 2022 validation cohort comprises 149 donors with paired TCRα+TCRβ immunosequencing (Zenodo DOI:10.5281/zenodo.5719516) and HLA ground truth available through HLAGuessr's training data (`HLA_grouped_patients.tsv`). Donor IDs are pure integers (e.g. `10270`) in the HLA file.

Ground truth is extracted for the 94 alleles modelable by HLAGuessr, enabling direct comparison between methods. Carrier rate across all patient-allele pairs is 9.5%, consistent with diploid HLA biology (each individual carries ~2 alleles per locus across 7 loci).

In [ ]:
hla_raw = pd.read_csv(HLA_FILE, sep='\t')

# Extract all unique patient IDs
all_ids = set()
for _, row in hla_raw.iterrows():
    for col in ['pos_patients','neg_patients']:
        all_ids.update([x.strip() for x in str(row[col]).split(',')])

# Russell donors = pure numeric IDs
nica_ids  = {x for x in all_ids if re.match(r'^\d+$', x)}
tcr_files = set(os.listdir(RUSSELL_DIR + 'TCRB/'))
file_ids  = {f.split('_run')[0].replace('nica_','') for f in tcr_files}
overlap   = nica_ids & file_ids  # donors with both HLA and TCR data

print(f'Russell donors in HLA file:         {len(nica_ids)}')
print(f'Russell donors with TCR files:      {len(file_ids)}')
print(f'Donors with both HLA + TCR:         {len(overlap)}')

# Build ground truth table for 94 modelable alleles
modelable_alleles = sorted(pd.read_csv(CROSSVAL_CSV)['HLA'].unique().tolist())
rows = []
for _, row in hla_raw.iterrows():
    if row['HLA'] not in modelable_alleles: continue
    pos = {x.strip() for x in str(row['pos_patients']).split(',')}
    for pid in overlap:
        rows.append({'patient_id': pid, 'HLA': row['HLA'],
                     'true_carrier': pid in pos})

ground_truth = pd.DataFrame(rows)
print(f'\nGround truth shape: {ground_truth.shape}')
print(f'Carrier rate:       {ground_truth["true_carrier"].mean()*100:.1f}%')
print(f'Modelable alleles:  {len(modelable_alleles)}')

## 2. Repertoire loading & QC

TCRβ files from Zenodo are tab-separated with columns: `cdr3`, `v_gene` (IMGT full notation with allele, e.g. `TRBV5-1*01`), `j_gene`, `productive`. Only productive clonotypes with canonical CDR3 sequences are retained.

V-genes are normalized to THNet format (full gene name without allele: `TRBV5-1*01` → `TRBV5-1`). Two pseudogenes (`TRBV20/OR9-2`, `TRBV26/OR9-2`) absent from THNet's training vocabulary are removed (5 clones total, 0.000% of data).

**QC filter**: minimum 1,000 productive clonotypes per chain. 20 donors fail this threshold (median 300 clones), leaving **129 QC-pass donors** for analysis.

In [ ]:
mp = Model_prediction()
hi = HLA_inference()
thnet_vgenes = set(mp.v_gene_list)

print(f'THNet alleles: {len(hi.hla_list)} | V-genes: {len(thnet_vgenes)}')

beta_frames = []
for pid in sorted(overlap):
    path = RUSSELL_DIR + f'TCRB/nica_{pid}_run4_umi5_B.tsv'
    if not os.path.exists(path): continue
    df = pd.read_csv(path, sep='\t', usecols=['cdr3','v_gene','productive'])
    df = df[df['productive']==True].dropna(subset=['cdr3','v_gene'])
    df = df[df['cdr3'].str.match(r'^C[A-Z]+F$|^C[A-Z]+[WY]$')]
    df['patient_id']    = pid
    df['v_gene_thnet']  = df['v_gene'].apply(norm_vgene_thnet)
    beta_frames.append(df)

beta = pd.concat(beta_frames, ignore_index=True)

# V-gene compatibility
invalid_vgenes = set(beta['v_gene_thnet'].dropna().unique()) - thnet_vgenes
print(f'Invalid V-genes for THNet: {sorted(invalid_vgenes)}')
beta_clean = beta[beta['v_gene_thnet'].isin(thnet_vgenes)].copy()

# QC filter
MIN_CLONES = 1000
clones_per_pat = beta_clean.groupby('patient_id').size()
qc_pass = clones_per_pat[clones_per_pat >= MIN_CLONES].index.tolist()

print(f'\nTCRβ clones (all):    {len(beta_clean):,}')
print(f'Donors passing QC:    {len(qc_pass)}/149')
print(f'Donors failing QC:    {149 - len(qc_pass)} (< {MIN_CLONES} clones)')

## 3. THNet inference — 129 donors × 208 alleles

THNet is applied to all 129 QC-pass donors. Predictions are cached per donor to allow resumption. Runtime: ~17 minutes on a server with RTX 4060 GPU (single-threaded Python API).

**Note on DPA1*01:03**: as observed in the Rosati cohort, this allele is predicted True in 100% of donors regardless of repertoire content — a model artefact caused by class imbalance in THNet training (~95% DPA1*01:03 frequency in European training donors). Excluded from all downstream analyses.

In [ ]:
results_thnet = []
t0 = time.time()

print(f'Running THNet — {len(qc_pass)} donors × 208 alleles')

for i, pid in enumerate(qc_pass):
    out_file = RUSSELL_THNET_OUT + str(pid) + '_thnet.csv'

    if os.path.exists(out_file):
        results_thnet.append(pd.read_csv(out_file))
        continue

    pat_df = beta_clean[beta_clean['patient_id']==pid][['cdr3','v_gene_thnet']]\
             .rename(columns={'v_gene_thnet':'v_gene'})
    pat_df['sample'] = str(pid)

    try:
        pred_raw = mp.Get_prediction(pat_df)
        scores   = pred_raw[str(pid)]
        calls    = hi.convert_pred(list(scores.values()))
        df_pat   = pd.DataFrame({
            'patient_id':        str(pid),
            'HLA':               [a.replace('HLA-','') for a in scores.keys()],
            'score':             list(scores.values()),
            'predicted_carrier': calls.astype(bool)
        })
        df_pat.to_csv(out_file, index=False)
        results_thnet.append(df_pat)
    except Exception as e:
        print(f'ERROR {pid}: {e}')

    elapsed = time.time() - t0
    eta     = elapsed / (i+1) * (len(qc_pass) - i - 1)
    if (i+1) % 20 == 0:
        print(f'{i+1}/{len(qc_pass)} | {elapsed/60:.1f} min | ETA {eta/60:.1f} min')

thnet_all = pd.concat(results_thnet, ignore_index=True)
thnet_all.to_csv(RESULTS + 'thnet_russell_predictions.csv', index=False)

print(f'\nDone in {(time.time()-t0)/60:.1f} min')
print(f'Total predictions: {len(thnet_all):,} | Donors: {thnet_all["patient_id"].nunique()}')

## 4. Results overview

DPA1*01:03 is excluded throughout. The overall carrier rate after applying THNet's default threshold (score ≥ 0.9005) is 1.2% — lower than the ground truth carrier rate of 9.5%. This reflects the conservative nature of the threshold applied to a population underrepresented in training.

In [ ]:
thnet = pd.read_csv(RESULTS + 'thnet_russell_predictions.csv')
thnet_clean = thnet[thnet['HLA'] != 'DPA1*01:03'].copy()
thnet_clean['patient_id'] = thnet_clean['patient_id'].astype(str)
ground_truth['patient_id'] = ground_truth['patient_id'].astype(str)

print(f'Total predictions (excl. DPA1*01:03): {len(thnet_clean):,}')
print(f'Donors: {thnet_clean["patient_id"].nunique()}')
print(f'Carrier rate (threshold 0.9005): {thnet_clean["predicted_carrier"].mean()*100:.1f}%')
print(f'Ground truth carrier rate:       9.5%')
print()
print('Top 15 alleles by predicted carriers:')
top = thnet_clean[thnet_clean['predicted_carrier']==True]\
      .groupby('HLA').agg(n=('patient_id','count'), mean_score=('score','mean'))\
      .sort_values('n', ascending=False).head(15)
top['mean_score'] = top['mean_score'].round(3)
print(top.to_string())

## 5. Quantitative validation against ground truth

Validation is performed on the 94 alleles shared between THNet and HLAGuessr (the modelable allele set). DPA1*01:03 is excluded.

**Key findings**:
- **Specificity 99.6%**: the model almost never generates false positives — when it predicts non-carrier, it is almost always correct
- **Sensitivity 12.7%**: only 1 in 8 true carriers is detected — driven by population mismatch
- **PPV 76.0%**: when the model predicts True, it is correct 76% of the time
- **AUC-ROC 0.739**: the raw score discriminates carriers from non-carriers meaningfully, even in this underrepresented population

In [ ]:
shared_alleles = sorted(
    set(thnet_clean['HLA'].unique()) &
    set(ground_truth['HLA'].unique()) - {'DPA1*01:03'}
)
print(f'Shared alleles for validation: {len(shared_alleles)}')

thnet_val = thnet_clean[thnet_clean['HLA'].isin(shared_alleles)]\
            .merge(ground_truth, on=['patient_id','HLA'], how='inner')

y_true  = thnet_val['true_carrier'].astype(int)
y_pred  = thnet_val['predicted_carrier'].astype(int)
y_score = thnet_val['score']

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

print(f'\n=== OVERALL VALIDATION METRICS ===')
print(f'TP={tp} | TN={tn} | FP={fp} | FN={fn}')
print(f'Accuracy:    {accuracy_score(y_true, y_pred)*100:.1f}%')
print(f'Sensitivity: {tp/(tp+fn)*100:.1f}%  (carriers detected)')
print(f'Specificity: {tn/(tn+fp)*100:.1f}%  (non-carriers correctly rejected)')
print(f'PPV:         {tp/(tp+fp)*100:.1f}%  (precision when predicting True)')
print(f'AUC-ROC:     {roc_auc_score(y_true, y_score):.3f}')

# Per allele
allele_metrics = []
for hla in shared_alleles:
    sub = thnet_val[thnet_val['HLA']==hla]
    if sub['true_carrier'].sum() < 3: continue
    try:
        auc = roc_auc_score(sub['true_carrier'], sub['score'])
        yt  = sub['true_carrier'].astype(int)
        yp  = sub['predicted_carrier'].astype(int)
        tn_,fp_,fn_,tp_ = confusion_matrix(yt,yp).ravel()
        allele_metrics.append({
            'HLA':             hla,
            'n_true_carriers': int(sub['true_carrier'].sum()),
            'n_predicted':     int(sub['predicted_carrier'].sum()),
            'AUC':             round(auc,3),
            'sensitivity_%':   round(tp_/(tp_+fn_)*100,1) if (tp_+fn_)>0 else 0,
            'specificity_%':   round(tn_/(tn_+fp_)*100,1) if (tn_+fp_)>0 else 0,
        })
    except: pass

df_metrics = pd.DataFrame(allele_metrics).sort_values('AUC', ascending=False)
print(f'\n=== PER-ALLELE AUC (top 15) ===')
print(df_metrics.head(15).to_string(index=False))

## 6. High-confidence predictions (≥90%)

Restricting to predictions where score ≥ 90% yields a high-confidence subset: **31/129 donors (24%)** have at least one allele confirmed at this threshold. At this stringency, PPV rises to **87.8%** — when THNet is highly confident, it is almost always correct.

This subset is the most reliable output for downstream analyses involving this cohort.

In [ ]:
thnet_90 = thnet_val[
    (thnet_val['predicted_carrier']==True) &
    (thnet_val['score'] >= 0.90)
]

tp_90 = thnet_90[thnet_90['true_carrier']==True]
fp_90 = thnet_90[thnet_90['true_carrier']==False]

print(f'Predictions at ≥90%:              {len(thnet_90)}')
print(f'Donors with ≥1 allele at ≥90%:   {thnet_90["patient_id"].nunique()}/129')
print(f'TP: {len(tp_90)} | FP: {len(fp_90)}')
print(f'PPV at ≥90%: {len(tp_90)/len(thnet_90)*100:.1f}%')
print()
print('Alleles with predictions at ≥90%:')
allele_90 = thnet_90.groupby('HLA').agg(
    n_total=('patient_id','count'),
    n_true=('true_carrier','sum')
).sort_values('n_total', ascending=False)
allele_90['PPV_%'] = (allele_90['n_true']/allele_90['n_total']*100).round(1)
print(allele_90.to_string())

## 7. Population mismatch analysis

The systematic underdetection (sensitivity 12.7%) is explained by the training-test population mismatch. THNet was trained on European/American cohorts where HLA-associated public clonotypes were identified. Nicaraguan donors carry different HLA allele frequencies and different public clonotype repertoires.

Evidence: alleles common in Latin American populations (e.g. DQA1*03:01 — carrier rate 45.6% in this cohort) have near-zero sensitivity (0%), while alleles shared across populations (e.g. A*02:01, DQA1*05:01) retain meaningful AUC values.

In [ ]:
# Compare true carrier rates vs THNet detection rates
gt_rates    = ground_truth.groupby('HLA')['true_carrier'].mean().rename('true_carrier_rate')
thnet_rates = thnet_val.groupby('HLA')['predicted_carrier'].mean().rename('thnet_predicted_rate')

comparison = pd.concat([gt_rates, thnet_rates], axis=1).dropna()
comparison['detection_ratio'] = (comparison['thnet_predicted_rate'] /
                                  comparison['true_carrier_rate']).round(3)
comparison = comparison.sort_values('true_carrier_rate', ascending=False)

print('True vs predicted carrier rates (top 15 most common alleles):')
print(comparison.head(15).round(3).to_string())
print()

# Locus-level AUC summary
df_metrics['locus'] = df_metrics['HLA'].apply(lambda x: x.split('*')[0])
locus_auc = df_metrics.groupby('locus')['AUC'].agg(['mean','count'])\
            .sort_values('mean', ascending=False)
print('Mean AUC by locus:')
print(locus_auc.round(3).to_string())

## 8. Conclusions

- **THNet successfully ran on 129/149 QC-pass Russell donors** producing 26,703 predictions across 207 alleles (DPA1*01:03 excluded as artefact).

- **AUC-ROC = 0.739**: THNet's raw scores meaningfully discriminate HLA carriers from non-carriers in a Nicaraguan cohort, despite being trained predominantly on European/American populations.

- **High sensitivity-specificity tradeoff**: at the default threshold (0.9005), specificity is 99.6% and PPV is 76% — the model is conservative and reliable when it does predict. Sensitivity is 12.7%, driven by population mismatch.

- **High-confidence subset (≥90%)**: 31/129 donors (24%) have at least one allele confirmed with PPV = 87.8%. Several alleles achieve 100% PPV at this threshold (A*02:01, DPB1*04:01, C*07:02, A*01:01).

- **Population mismatch is the central limitation**: alleles common in Latin American populations (DQA1*03:01 — 45.6% carrier rate) are completely missed (sensitivity 0%), while alleles shared across populations retain meaningful detection.

- **Key recommendation**: for Nicaraguan or Latin American cohorts, THNet predictions should be interpreted using raw scores (AUC-based) rather than the binary threshold calibrated for European populations. Models retrained on diverse-ancestry cohorts are needed for equitable performance.

- **DPA1*01:03 artefact**: predicted True in 100% of donors — excluded from all analyses. Same artefact observed in Rosati cohort.